In [1]:
import os
import pandas as pd
from pyproj import Transformer

# ----------------- CONFIGURACIÓN DE RUTAS -----------------
DIR_DATOS = r"C:\Proyectos_de_codigo\Universidad\Sismo\Sismo-geotecnia\datos\datos depurados"

ARCHIVOS = [
    "Sismos_260_mw_def.csv",
    "Sismos_50_mw_def.csv"
]

# Inicializar transformador de coordenadas:
# EPSG:4326  -> WGS 84 (Latitud, Longitud)
# EPSG:9377  -> MAGNA-SIRGAS / Origen Nacional CTM12 (Este, Norte)
# always_xy=True garantiza el orden de entrada/salida como (longitud, latitud) -> (X, Y)
transformador = Transformer.from_crs("EPSG:4326", "EPSG:9377", always_xy=True)

def procesar_y_convertir(nombre_archivo):
    ruta_entrada = os.path.join(DIR_DATOS, nombre_archivo)
    
    if not os.path.exists(ruta_entrada):
        print(f"❌ [ERROR] No se encontró el archivo: {ruta_entrada}")
        return

    print(f"Procesando: {nombre_archivo}...")
    df = pd.read_csv(ruta_entrada)

    # Identificar nombres de columnas de latitud, longitud y profundidad
    col_lat = "latitude" if "latitude" in df.columns else "latitud"
    col_lon = "longitude" if "longitude" in df.columns else "longitud"
    col_depth = "depth" if "depth" in df.columns else ("profundidad" if "profundidad" in df.columns else "depth_km")

    # Extraer únicamente las columnas solicitadas
    df_filtrado = df[[col_lat, col_lon, col_depth]].copy()
    
    # Renombrar a nombres estándar temporales
    df_filtrado.columns = ["latitud", "longitud", "profundidad"]
    
    # Eliminar posibles valores nulos
    df_filtrado = df_filtrado.dropna()

    # Reproyección a CTM12 (EPSG:9377)
    # Importante: pyproj con always_xy=True recibe (Longitud, Latitud)
    x_este, y_norte = transformador.transform(
        df_filtrado["longitud"].values, 
        df_filtrado["latitud"].values
    )

    # Crear la nueva estructura de datos solicitada
    df_resultado = pd.DataFrame({
        "latitud": df_filtrado["latitud"],
        "longitud": df_filtrado["longitud"],
        "X": x_este,
        "Y": y_norte,
        "Z": df_filtrado["profundidad"]
    })

    # Generar el nombre del nuevo archivo terminado en _CTM12.csv
    nombre_base, ext = os.path.splitext(nombre_archivo)
    nombre_salida = f"{nombre_base}_CTM12{ext}"
    ruta_salida = os.path.join(DIR_DATOS, nombre_salida)

    # Guardar en CSV
    df_resultado.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"✅ Archivo generado exitosamente: {ruta_salida}\n")

# ==============================================================================
# EJECUCIÓN DEL PROCESO
# ==============================================================================

print("Iniciando la conversión de coordenadas a CTM12 (EPSG:9377)...\n")

for archivo in ARCHIVOS:
    procesar_y_convertir(archivo)

print("¡Proceso finalizado correctamente!")

Iniciando la conversión de coordenadas a CTM12 (EPSG:9377)...

Procesando: Sismos_260_mw_def.csv...
✅ Archivo generado exitosamente: C:\Proyectos_de_codigo\Universidad\Sismo\Sismo-geotecnia\datos\datos depurados\Sismos_260_mw_def_CTM12.csv

Procesando: Sismos_50_mw_def.csv...
✅ Archivo generado exitosamente: C:\Proyectos_de_codigo\Universidad\Sismo\Sismo-geotecnia\datos\datos depurados\Sismos_50_mw_def_CTM12.csv

¡Proceso finalizado correctamente!
